# Demo 2: Kubernetes Auth Method con Vault en Docker Desktop

En este notebook, una aplicación en el namespace `app` usa su **ServiceAccount** para autenticarse en Vault mediante el **Kubernetes Auth Method** y leer un secreto.

Requisitos:

- Haber ejecutado el notebook `1_vault_basico_docker_desktop.ipynb`.
- Tener a mano el `VAULT_TOKEN` (token raíz) obtenido de los logs de Vault.


In [ ]:
%%bash
set -euo pipefail

NAMESPACE_VAULT=vault
NAMESPACE_APP=app

echo "==> Creando namespace de la app ($NAMESPACE_APP) si no existe"
kubectl create namespace "$NAMESPACE_APP" 2>/dev/null || echo "Namespace ya existe"

echo "==> Creando pod helper temporal en el namespace de Vault"
kubectl -n "$NAMESPACE_VAULT" run helper --image=bitnami/kubectl:latest --restart=Never -- sleep 3600 2>/dev/null || echo "helper ya existe"

echo "==> Obteniendo CA y token del ServiceAccount por defecto"
kubectl -n "$NAMESPACE_VAULT" exec helper -- cat /var/run/secrets/kubernetes.io/serviceaccount/ca.crt > ca.crt
KUBE_TOKEN=$(kubectl -n "$NAMESPACE_VAULT" exec helper -- cat /var/run/secrets/kubernetes.io/serviceaccount/token)
KUBE_HOST=$(kubectl config view --minify -o jsonpath='{.clusters[0].cluster.server}')

echo "KUBE_HOST=$KUBE_HOST"

echo "==> Recuerda exportar VAULT_ADDR y VAULT_TOKEN en tu entorno antes de seguir"
echo "    export VAULT_ADDR=http://127.0.0.1:8200"
echo "    export VAULT_TOKEN=<TU_ROOT_TOKEN>"


In [ ]:
%%bash
set -euo pipefail

: "${VAULT_ADDR:?Debes exportar VAULT_ADDR}"
: "${VAULT_TOKEN:?Debes exportar VAULT_TOKEN}"

echo "==> Habilitando Kubernetes Auth en Vault (si no estaba habilitado)"
vault auth enable kubernetes 2>/dev/null || echo "kubernetes auth ya habilitado"

echo "==> Configurando Kubernetes Auth en Vault"
vault write auth/kubernetes/config \
  token_reviewer_jwt=@<(kubectl -n vault exec helper -- cat /var/run/secrets/kubernetes.io/serviceaccount/token) \
  kubernetes_host="$(kubectl config view --minify -o jsonpath='{.clusters[0].cluster.server}')" \
  kubernetes_ca_cert=@ca.crt

echo "==> Creando secreto en Vault y policy"
vault kv put secret/app/config username="demo-user" password="demo-pass"

vault policy write app-policy - <<EOF
path "secret/data/app/config" {
  capabilities = ["read"]
}
EOF

vault write auth/kubernetes/role/app-role \
  bound_service_account_names=app-sa \
  bound_service_account_namespaces=app \
  policies=app-policy \
  ttl=1h


In [ ]:
%%bash
set -euo pipefail

cat <<EOF | kubectl apply -f -
apiVersion: v1
kind: ServiceAccount
metadata:
  name: app-sa
  namespace: app
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vault-demo-app
  namespace: app
spec:
  replicas: 1
  selector:
    matchLabels:
      app: vault-demo-app
  template:
    metadata:
      labels:
        app: vault-demo-app
    spec:
      serviceAccountName: app-sa
      containers:
      - name: app
        image: curlimages/curl:8.8.0
        command: ["/bin/sh","-c"]
        args:
        - |
          echo "Obteniendo token del ServiceAccount...";
          TOKEN=$(cat /var/run/secrets/kubernetes.io/serviceaccount/token);
          echo "Pidiendo token de Vault...";
          RESP=$(curl -s --request POST \
            --data "{\"jwt\": \"$TOKEN\", \"role\": \"app-role\"}" \
            $VAULT_ADDR/v1/auth/kubernetes/login);
          VAULT_TOKEN=$(echo "$RESP" | jq -r .auth.client_token);
          echo "Token de Vault: $VAULT_TOKEN";
          echo "Leyendo secreto...";
          curl -s --header "X-Vault-Token: $VAULT_TOKEN" \
            $VAULT_ADDR/v1/secret/data/app/config | jq .;
          echo "Listo. Durmiendo...";
          sleep 3600;
        env:
        - name: VAULT_ADDR
          value: "http://vault.vault.svc.cluster.local:8200"
EOF

kubectl -n app get pods


## Ver logs de la aplicación

Ejecuta en una celda separada o una terminal:

```bash
kubectl -n app logs deploy/vault-demo-app
```

Deberías ver cómo la app:

1. Obtiene el token JWT del ServiceAccount del Pod.
2. Lo envía al endpoint `auth/kubernetes/login` de Vault.
3. Recibe un `client_token` de Vault.
4. Usa ese token para leer `secret/data/app/config` y mostrar el JSON del secreto.
